# Anomaly detection with PyOD

Notebook version of [AnomalyDetectionUsingPyOD](https://github.com/JaminJeong/AnomalyDetectionUsingPyOD) (daily minimum temperatures, sliding windows, multiple detectors). Temperatures are loaded with **`%%cribl_search`** using Cribl Search [`externaldata`](https://docs.cribl.io/search/externaldata/) (same public CSV as the upstream example—see [daily-min-temperatures.csv](https://raw.githubusercontent.com/jbrownlee/Datasets/master/daily-min-temperatures.csv) and [Machine Learning Mastery](https://machinelearningmastery.com/time-series-datasets-for-machine-learning/)). [PyOD](https://github.com/yzhao062/pyod) is installed in **Setup** via `micropip`.

## How to run

1. Run **Setup** once (downloads PyOD stack; first run can take several minutes).
2. Run **Shared preprocessing** (Python helpers and constants).
3. Run **Load temperatures** (`%%cribl_search` with `externaldata`) — requires this notebook to run **inside a hosted Cribl app** where Search is available (like other `%%cribl_search` examples).
4. Run **Materialize dataframe**, then **train/test windows**, then each **detector** cell (each draws its own figure). **Variational Auto Encoder** and **Auto Encoder** often **do not run** in Pyodide. The upstream **LOCI** screenshots are not part of the upstream script's active models; this notebook keeps all **18** detectors from `anomaly_detection_all_model.py`.

### Setup

Installs **SciPy**, **scikit-learn**, and **joblib**, then **PyOD** with `deps=False` (PyPI also declares **Numba**, which has no Pyodide wheel—models that need Numba are skipped). **Feature Bagging** needs PyPI **`combo`**, which has no Pyodide wheel here— that cell prints a skip. Requires PyPI/jsDelivr (allow-listed in `proxies.yml`).

In [ ]:
import micropip

# PyPI declares `numba` for pyod, but Numba has no Pyodide wheel — install the
# scientific stack explicitly, then pyod without transitive deps.
await micropip.install(['scipy', 'scikit-learn', 'joblib'])
await micropip.install(['pyod'], deps=False)

### Shared preprocessing

Imports, constants (`window_size`, `contamination`), helper functions for sliding windows and plots, and the LOF list used by **LSCP**. Run **Setup** above first so `pyod` is available.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from pyod.models.lof import LOF

window_size = 30
contamination = 0.25
outliers_fraction = contamination
random_state = 42


def make_data_sampling(data, window_size):
    n_samples = len(data)
    if n_samples < window_size:
        raise ValueError('window size is too long for series')
    return np.array([data[i : i + window_size] for i in range(n_samples - window_size + 1)])


def plot_anomalies(title, real_value, y_pred):
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(14, 3.5))
    ax.plot(np.arange(len(real_value)), real_value, label='real')
    scatter_x = [i for i, lab in enumerate(y_pred) if lab == 1]
    scatter_y = [real_value[i] for i in scatter_x]
    ax.scatter(scatter_x, scatter_y, color='0.5', edgecolor='r', label='anomaly')
    ax.set_title(title)
    ax.legend(loc='upper right')
    plt.tight_layout()
    plt.show()
    plt.close(fig)


def run_detector(title, clf):
    clf.fit(df_data_train)
    y_pred = clf.predict(df_data_test)
    real_value = df_data_test[:, -1]
    plot_anomalies(title, real_value, y_pred)


### Load temperatures (`externaldata`)

Fetches the CSV through **[`externaldata`](https://docs.cribl.io/search/externaldata/)** so retrieval happens in **Cribl Search**, not via `pd.read_csv` in Pyodide. Uses stock datatype **`generic_csv`**. If Search rejects the datatype on your tenant, try **`CSV`** (v1 stock) or your admin's comma-separated datatype.

In [ ]:
%%cribl_search var=temp_df lang=kql limit=0 preview=false earliest=-50y latest=now
externaldata
[
  "https://raw.githubusercontent.com/jbrownlee/Datasets/master/daily-min-temperatures.csv"
]
with(
  datatype="generic_csv"
)

In [ ]:
import io

import pandas as pd

# externaldata + generic_csv often lands each CSV row in one Search field (e.g. `Event` / `_raw`),
# not as separate Date/Temp columns — rebuild the CSV text then parse.
_cols_lower = {str(c).strip().lower(): c for c in temp_df.columns}
_raw_col = None
for key in ('event', '_raw'):
    if key in _cols_lower:
        _raw_col = _cols_lower[key]
        break
if _raw_col is None:
    for c in temp_df.columns:
        if str(c).startswith('_'):
            _raw_col = c
            break
if _raw_col is None:
    non_time = [c for c in temp_df.columns if str(c).lower() not in ('time', '_time')]
    _raw_col = non_time[-1] if non_time else temp_df.columns[-1]

_lines = temp_df[_raw_col].astype(str).str.strip()
_lines = _lines[_lines.str.len() > 0]
# Drop repeated header rows (some tenants echo the CSV header per event)
_hdr = '"Date","Temp"'
_lines = _lines[_lines != _hdr]
_csv_text = '\n'.join([_hdr] + _lines.tolist())
df_data = pd.read_csv(io.StringIO(_csv_text))
df_data.columns = [str(c).strip() for c in df_data.columns]
if 'Temp' not in df_data.columns:
    raise ValueError(f'Expected Temp column after CSV parse; got {list(df_data.columns)}')
df_data['Temp'] = pd.to_numeric(df_data['Temp'], errors='coerce')
df_data = df_data.dropna(subset=['Temp']).reset_index(drop=True)
df_data.head()

In [ ]:
raw_train, raw_test = train_test_split(
    np.array(df_data['Temp']), test_size=0.2, shuffle=False
)

n_train_windows = len(raw_train) - window_size + 1
if n_train_windows < 2:
    raise ValueError(f'Not enough training points for window_size={window_size}; got {len(raw_train)} temps')

# sklearn / pyod require n_neighbors < n_samples (training rows are sliding windows)
_nn_cap = max(1, n_train_windows - 1)


def _cap_nn(k: int) -> int:
    return max(1, min(int(k), _nn_cap))


LOF_N = _cap_nn(35)
COF_N = LOF_N

_nn_grid = (5, 10, 15, 20, 25, 30, 35, 40, 45, 50)
detector_list = []
_seen = set()
for k in _nn_grid:
    v = _cap_nn(k)
    if v not in _seen:
        _seen.add(v)
        detector_list.append(LOF(n_neighbors=v))
if len(detector_list) < 2:
    detector_list = [LOF(n_neighbors=_nn_cap), LOF(n_neighbors=max(1, _nn_cap // 2))]

df_data_train = make_data_sampling(raw_train, window_size)
df_data_test = make_data_sampling(raw_test, window_size)
print('train windows', df_data_train.shape, 'test windows', df_data_test.shape, 'LOF_N', LOF_N, 'detectors for LSCP', len(detector_list))

### Train series (subsampled)

Same idea as `GenGraph('train')` in the upstream `save_graph.py`.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(raw_train[::30])
ax.set_title('Train temperatures (every 30th point)')
plt.tight_layout()
plt.show()
plt.close(fig)

### Test series (subsampled)

Same idea as `GenGraph('test')`.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(raw_test[::30])
ax.set_title('Test temperatures (every 30th point)')
plt.tight_layout()
plt.show()
plt.close(fig)

## Detectors

Each section fits **one** model on `df_data_train`, scores `df_data_test`, and plots the **last timestep** of each window against anomalies (matching `BasicGenGraph` in the upstream project).

### Angle-based Outlier Detector (ABOD)

In [ ]:
title = 'Angle-based Outlier Detector (ABOD)'
try:
    from pyod.models.abod import ABOD
    clf = ABOD(contamination=outliers_fraction)
    run_detector(title, clf)
except Exception as err:
    print(f'{title}: skipped  -  {type(err).__name__}: {err}')

### Cluster-based Local Outlier Factor (CBLOF)

In [ ]:
title = 'Cluster-based Local Outlier Factor (CBLOF)'
try:
    from pyod.models.cblof import CBLOF
    clf = CBLOF(contamination=outliers_fraction, check_estimator=False, random_state=random_state, n_clusters=15)
    run_detector(title, clf)
except Exception as err:
    print(f'{title}: skipped  -  {type(err).__name__}: {err}')

### Feature Bagging

In [ ]:
title = 'Feature Bagging'
try:
    from pyod.models.lof import LOF
    from pyod.models.feature_bagging import FeatureBagging
    clf = FeatureBagging(LOF(n_neighbors=LOF_N), contamination=outliers_fraction, random_state=random_state)
    run_detector(title, clf)
except Exception as err:
    print(f'{title}: skipped  -  {type(err).__name__}: {err}')

### Histogram-base Outlier Detection (HBOS)

In [ ]:
title = 'Histogram-base Outlier Detection (HBOS)'
try:
    from pyod.models.hbos import HBOS
    clf = HBOS(contamination=outliers_fraction)
    run_detector(title, clf)
except Exception as err:
    print(f'{title}: skipped  -  {type(err).__name__}: {err}')

### Isolation Forest

In [ ]:
title = 'Isolation Forest'
try:
    from pyod.models.iforest import IForest
    clf = IForest(contamination=outliers_fraction, random_state=random_state)
    run_detector(title, clf)
except Exception as err:
    print(f'{title}: skipped  -  {type(err).__name__}: {err}')

### K Nearest Neighbors (KNN)

In [ ]:
title = 'K Nearest Neighbors (KNN)'
try:
    from pyod.models.knn import KNN
    clf = KNN(contamination=outliers_fraction)
    run_detector(title, clf)
except Exception as err:
    print(f'{title}: skipped  -  {type(err).__name__}: {err}')

### Average KNN

In [ ]:
title = 'Average KNN'
try:
    from pyod.models.knn import KNN
    clf = KNN(method='mean', contamination=outliers_fraction)
    run_detector(title, clf)
except Exception as err:
    print(f'{title}: skipped  -  {type(err).__name__}: {err}')

### Median KNN

In [ ]:
title = 'Median KNN'
try:
    from pyod.models.knn import KNN
    clf = KNN(method='median', contamination=outliers_fraction)
    run_detector(title, clf)
except Exception as err:
    print(f'{title}: skipped  -  {type(err).__name__}: {err}')

### Local Outlier Factor (LOF)

In [ ]:
title = 'Local Outlier Factor (LOF)'
try:
    from pyod.models.lof import LOF
    clf = LOF(n_neighbors=LOF_N, contamination=outliers_fraction)
    run_detector(title, clf)
except Exception as err:
    print(f'{title}: skipped  -  {type(err).__name__}: {err}')

### Minimum Covariance Determinant (MCD)

In [ ]:
title = 'Minimum Covariance Determinant (MCD)'
try:
    from pyod.models.mcd import MCD
    clf = MCD(contamination=outliers_fraction, random_state=random_state)
    run_detector(title, clf)
except Exception as err:
    print(f'{title}: skipped  -  {type(err).__name__}: {err}')

### One-class SVM (OCSVM)

In [ ]:
title = 'One-class SVM (OCSVM)'
try:
    from pyod.models.ocsvm import OCSVM
    clf = OCSVM(contamination=outliers_fraction)
    run_detector(title, clf)
except Exception as err:
    print(f'{title}: skipped  -  {type(err).__name__}: {err}')

### Principal Component Analysis (PCA)

In [ ]:
title = 'Principal Component Analysis (PCA)'
try:
    from pyod.models.pca import PCA
    clf = PCA(contamination=outliers_fraction, random_state=random_state)
    run_detector(title, clf)
except Exception as err:
    print(f'{title}: skipped  -  {type(err).__name__}: {err}')

### Stochastic Outlier Selection (SOS)

In [ ]:
title = 'Stochastic Outlier Selection (SOS)'
try:
    from pyod.models.sos import SOS
    clf = SOS(contamination=outliers_fraction)
    run_detector(title, clf)
except Exception as err:
    print(f'{title}: skipped  -  {type(err).__name__}: {err}')

### Locally Selective Combination (LSCP)

In [ ]:
title = 'Locally Selective Combination (LSCP)'
try:
    from pyod.models.lscp import LSCP
    clf = LSCP(detector_list, contamination=outliers_fraction, random_state=random_state)
    run_detector(title, clf)
except Exception as err:
    print(f'{title}: skipped  -  {type(err).__name__}: {err}')

### Connectivity-Based Outlier Factor (COF)

In [ ]:
title = 'Connectivity-Based Outlier Factor (COF)'
try:
    from pyod.models.cof import COF
    clf = COF(n_neighbors=COF_N, contamination=outliers_fraction)
    run_detector(title, clf)
except Exception as err:
    print(f'{title}: skipped  -  {type(err).__name__}: {err}')

### Subspace Outlier Detection (SOD)

In [ ]:
title = 'Subspace Outlier Detection (SOD)'
try:
    from pyod.models.sod import SOD
    clf = SOD(contamination=outliers_fraction)
    run_detector(title, clf)
except Exception as err:
    print(f'{title}: skipped  -  {type(err).__name__}: {err}')

### Variational Auto Encoder

In [ ]:
title = 'Variational Auto Encoder'
try:
    from pyod.models.vae import VAE
    clf = VAE(epochs=15, contamination=contamination, gamma=0.8, capacity=0.2, batch_size=window_size)
    run_detector(title, clf)
except Exception as err:
    print(f'{title}: skipped  -  {type(err).__name__}: {err}')

### Auto Encoder

In [ ]:
title = 'Auto Encoder'
try:
    from pyod.models.auto_encoder import AutoEncoder
    clf = AutoEncoder(epochs=15, contamination=contamination, batch_size=window_size)
    run_detector(title, clf)
except Exception as err:
    print(f'{title}: skipped  -  {type(err).__name__}: {err}')

## Next steps

Tune `window_size` and `contamination`, try your own series, or export scores with `clf.decision_function(df_data_test)` once a model fits.